# 🎯 Multi-Touch Attribution (MTA) — End-to-End Python Implementation

> **Author:** Shashank Paliwal | Data Science Manager  
> **Domain:** Retail / CPG | Marketing Effectiveness  
> **Stack:** Python · NumPy · Pandas · Matplotlib · Seaborn  
> **Related:** [Market Mix Modeling Notebook](https://github.com/spaliwa1/marketing-analytics-portfolio/blob/40d77beb7bdd468fa576dacfa043c5c21270c729/market_mix_modeling.ipynb)

---

## Overview

Multi-Touch Attribution (MTA) answers a deceptively simple question:

> *A customer sees a display ad on Monday, clicks a search ad on Wednesday, opens an email on Thursday, and converts via a loyalty coupon on Friday. How much credit does each touchpoint deserve?*

This notebook builds a **complete MTA framework** across 8 channels (4 offsite, 4 onsite) — mirroring real-world implementations for large UK grocery retailers. We implement and compare:

| Model | Type | Key Idea |
|-------|------|----------|
| Last Touch | Rule-based | 100% credit to final touchpoint |
| First Touch | Rule-based | 100% credit to first touchpoint |
| Linear | Rule-based | Equal credit across all touchpoints |
| Time Decay | Rule-based | More credit to recent touchpoints |
| Shapley Value | Data-driven | Game-theoretic marginal contribution |
| Markov Chain | Data-driven | Transition probabilities + removal effect |

### Key Sections
1. [Customer Journey Simulation](#1-customer-journey-simulation)
2. [Exploratory Analysis](#2-exploratory-analysis)
3. [Rule-Based Attribution Models](#3-rule-based-attribution-models)
4. [Shapley Value Attribution](#4-shapley-value-attribution)
5. [Markov Chain Attribution](#5-markov-chain-attribution)
6. [Model Comparison](#6-model-comparison)
7. [Budget Implications](#7-budget-implications)
8. [Key Takeaways](#8-key-takeaways)

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from itertools import combinations, permutations
from collections import defaultdict
from math import factorial
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans'
})
PALETTE = ['#1B4F8A','#E05A2B','#2E9C6A','#9B59B6',
           '#F39C12','#1ABC9C','#E74C3C','#3498DB']
np.random.seed(42)
print('Setup complete ✅')

---
## 1. Customer Journey Simulation

We simulate **50,000 customer journeys** across 8 channels — 4 offsite (TV, Radio, OOH, Digital Display) and 4 onsite (In-store Display, Loyalty Coupon, Email, Leaflet).

Each journey:
- Has 1–6 touchpoints drawn from a realistic channel sequence
- Has a conversion probability influenced by channel combination and recency
- Has an order value for converted journeys

This mirrors the Tesco Clubcard ecosystem — where a customer might see a TV ad, receive a direct mail offer, open an email, and finally convert via a loyalty coupon in-store.

In [ ]:
CHANNELS = [
    'TV', 'Digital_Display', 'OOH', 'Radio',           # Offsite
    'Email', 'Loyalty_Coupon', 'Instore_Display', 'Leaflet'  # Onsite
]

# Channel-level properties
channel_props = {
    'TV':               {'awareness': 0.90, 'conversion': 0.04, 'avg_spend': 800},
    'Digital_Display':  {'awareness': 0.75, 'conversion': 0.08, 'avg_spend': 600},
    'OOH':              {'awareness': 0.65, 'conversion': 0.03, 'avg_spend': 250},
    'Radio':            {'awareness': 0.55, 'conversion': 0.03, 'avg_spend': 300},
    'Email':            {'awareness': 0.70, 'conversion': 0.12, 'avg_spend': 150},
    'Loyalty_Coupon':   {'awareness': 0.85, 'conversion': 0.18, 'avg_spend': 350},
    'Instore_Display':  {'awareness': 0.80, 'conversion': 0.15, 'avg_spend': 400},
    'Leaflet':          {'awareness': 0.60, 'conversion': 0.09, 'avg_spend': 200},
}

# Realistic channel sequence probabilities
# Offsite channels tend to appear early, onsite later
first_touch_probs = [0.30, 0.25, 0.12, 0.10, 0.08, 0.07, 0.05, 0.03]
subsequent_probs  = [0.10, 0.20, 0.08, 0.07, 0.18, 0.20, 0.12, 0.05]

def simulate_journey():
    n_touches = np.random.choice([1,2,3,4,5,6], p=[0.15,0.25,0.25,0.20,0.10,0.05])
    first = np.random.choice(CHANNELS, p=first_touch_probs)
    journey = [first]
    for _ in range(n_touches - 1):
        nxt = np.random.choice(CHANNELS, p=subsequent_probs)
        journey.append(nxt)

    # Conversion probability — increases with onsite channels and journey length
    base_conv = np.mean([channel_props[c]['conversion'] for c in journey])
    onsite_bonus = 0.05 * sum(1 for c in journey
                              if c in ['Loyalty_Coupon','Instore_Display','Email'])
    length_decay = max(0, 1 - 0.05 * len(journey))  # longer journeys slightly lower
    conv_prob = min(0.85, (base_conv + onsite_bonus) * length_decay)
    converted = np.random.random() < conv_prob
    order_value = np.random.lognormal(3.8, 0.5) if converted else 0  # ~£45 avg
    return journey, converted, round(order_value, 2)

N_JOURNEYS = 50000
records = [simulate_journey() for _ in range(N_JOURNEYS)]

df = pd.DataFrame(records, columns=['journey','converted','order_value'])
df['journey_length'] = df['journey'].apply(len)
df['journey_id'] = range(len(df))

conv_rate = df['converted'].mean()
total_rev  = df['order_value'].sum()
print(f'Total journeys   : {N_JOURNEYS:,}')
print(f'Conversions      : {df["converted"].sum():,} ({conv_rate:.1%})')
print(f'Total revenue    : £{total_rev:,.0f}')
print(f'Avg order value  : £{df[df["converted"]]["order_value"].mean():.2f}')
print(f'Avg journey len  : {df["journey_length"].mean():.1f} touchpoints')
df.head()

---
## 2. Exploratory Analysis

In [ ]:
# Touchpoint frequency
from collections import Counter
all_touches = [ch for journey in df['journey'] for ch in journey]
touch_counts = Counter(all_touches)

# Conversion rate by last touchpoint
df['last_touch'] = df['journey'].apply(lambda x: x[-1])
df['first_touch'] = df['journey'].apply(lambda x: x[0])
conv_by_last = df.groupby('last_touch')['converted'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Touchpoint frequency
ch_labels = [c.replace('_',' ') for c in touch_counts.keys()]
axes[0].barh(ch_labels, list(touch_counts.values()),
             color=PALETTE[:len(touch_counts)], edgecolor='white')
axes[0].set_title('Total Touchpoint Frequency', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Journey length distribution
axes[1].hist(df['journey_length'], bins=6, color=PALETTE[0],
             edgecolor='white', rwidth=0.8)
axes[1].set_title('Journey Length Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Touchpoints')
axes[1].set_ylabel('Number of Journeys')

# Conversion rate by last touch
colors_conv = [PALETTE[2] if v > conv_by_last.median() else PALETTE[6]
               for v in conv_by_last.values]
axes[2].barh([c.replace('_',' ') for c in conv_by_last.index],
             conv_by_last.values * 100,
             color=colors_conv, edgecolor='white')
axes[2].axvline(conv_rate * 100, color='navy', linestyle='--',
                linewidth=1.5, label=f'Overall avg ({conv_rate:.1%})')
axes[2].set_title('Conversion Rate by Last Touchpoint', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Conversion Rate (%)')
axes[2].invert_yaxis()
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('/home/claude/mta_eda.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Rule-Based Attribution Models

Rule-based models are simple and transparent but make strong, often wrong, assumptions.

| Model | Assumption | Problem |
|-------|-----------|--------|
| **Last Touch** | Final touchpoint drives conversion | Ignores all awareness-building |
| **First Touch** | First touchpoint drives conversion | Ignores all conversion-driving |
| **Linear** | All touchpoints equally important | Ignores channel effectiveness differences |
| **Time Decay** | Recent touchpoints matter more | Arbitrary decay function |

In [ ]:
converted_df = df[df['converted']].copy()

def last_touch_attribution(row):
    credits = {ch: 0.0 for ch in CHANNELS}
    credits[row['journey'][-1]] = row['order_value']
    return credits

def first_touch_attribution(row):
    credits = {ch: 0.0 for ch in CHANNELS}
    credits[row['journey'][0]] = row['order_value']
    return credits

def linear_attribution(row):
    credits = {ch: 0.0 for ch in CHANNELS}
    n = len(row['journey'])
    for ch in row['journey']:
        credits[ch] += row['order_value'] / n
    return credits

def time_decay_attribution(row, half_life=3):
    credits = {ch: 0.0 for ch in CHANNELS}
    journey = row['journey']
    n = len(journey)
    weights = np.array([2 ** ((i - n + 1) / half_life) for i in range(n)])
    weights /= weights.sum()
    for ch, w in zip(journey, weights):
        credits[ch] += row['order_value'] * w
    return credits

def aggregate_credits(func, df):
    result = {ch: 0.0 for ch in CHANNELS}
    for _, row in df.iterrows():
        for ch, val in func(row).items():
            result[ch] += val
    return result

last_touch  = aggregate_credits(last_touch_attribution, converted_df)
first_touch = aggregate_credits(first_touch_attribution, converted_df)
linear      = aggregate_credits(linear_attribution, converted_df)
time_decay  = aggregate_credits(time_decay_attribution, converted_df)

rule_df = pd.DataFrame({
    'Channel'    : [c.replace('_',' ') for c in CHANNELS],
    'Last Touch' : [last_touch[c] for c in CHANNELS],
    'First Touch': [first_touch[c] for c in CHANNELS],
    'Linear'     : [linear[c] for c in CHANNELS],
    'Time Decay' : [time_decay[c] for c in CHANNELS],
})

# Normalise to % share
for col in ['Last Touch','First Touch','Linear','Time Decay']:
    rule_df[col] = rule_df[col] / rule_df[col].sum() * 100

print(rule_df.round(1).to_string(index=False))

---
## 4. Shapley Value Attribution

**Shapley values** come from cooperative game theory. The idea: each channel's credit equals its **average marginal contribution** across all possible orderings of channels.

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(|N|-|S|-1)!}{|N|!} [v(S \cup \{i\}) - v(S)]$$

Where:
- **S** = subset of channels not including channel i
- **v(S)** = value (conversion rate) of the coalition S
- **φᵢ** = Shapley value (fair credit) for channel i

This is the **fairest** attribution model — it accounts for interactions between channels and gives each channel exactly what it contributes on the margin.

In [ ]:
# Build coalition value function v(S) = conversion rate of journeys containing exactly set S
# Approximation: use journeys where all channels in S appear

def coalition_value(channel_set, df):
    """Conversion rate for journeys containing all channels in channel_set."""
    if not channel_set:
        return df['converted'].mean()
    mask = df['journey'].apply(
        lambda j: all(c in j for c in channel_set)
    )
    subset = df[mask]
    if len(subset) < 10:  # too few journeys — return base rate
        return df['converted'].mean()
    return subset['converted'].mean()

def shapley_values(channels, df):
    """Compute Shapley values for all channels."""
    n = len(channels)
    shapley = {ch: 0.0 for ch in channels}

    for ch in channels:
        others = [c for c in channels if c != ch]
        for size in range(len(others) + 1):
            for subset in combinations(others, size):
                subset_set = list(subset)
                weight = (factorial(size) * factorial(n - size - 1)) / factorial(n)
                v_with    = coalition_value(subset_set + [ch], df)
                v_without = coalition_value(subset_set, df)
                shapley[ch] += weight * (v_with - v_without)
    return shapley

# Use top 5 channels for tractable exact computation
# (Full Shapley over 8 channels = 2^8 = 256 coalitions — feasible)
print('Computing Shapley values across all 8 channels...')
print('(This may take 1–2 minutes for 50K journeys)')
shapley = shapley_values(CHANNELS, df)

# Convert to revenue attribution
total_converted_rev = converted_df['order_value'].sum()
shapley_total = sum(shapley.values())
shapley_rev = {ch: (v / shapley_total) * total_converted_rev
               for ch, v in shapley.items()}

shapley_df = pd.DataFrame({
    'Channel': [c.replace('_',' ') for c in CHANNELS],
    'Shapley Value': [shapley[c] for c in CHANNELS],
    'Revenue Attribution (£)': [shapley_rev[c] for c in CHANNELS],
    '% Share': [shapley_rev[c] / total_converted_rev * 100 for c in CHANNELS]
}).sort_values('% Share', ascending=False)

print('\nShapley Value Attribution:')
print(shapley_df.round(2).to_string(index=False))

---
## 5. Markov Chain Attribution

**Markov Chain Attribution** models the customer journey as a sequence of states (channels) with transition probabilities. A channel's credit = its **removal effect**: how much does conversion probability drop if we remove this channel from all journeys?

**Steps:**
1. Build a **transition matrix** from journey sequences
2. Calculate **overall conversion probability** from the chain
3. For each channel, **remove it** and recalculate conversion probability
4. **Removal effect** = drop in conversion probability when channel is removed
5. Normalise removal effects to get attribution weights

In [ ]:
def build_transition_matrix(df):
    """Build channel transition counts from all journeys."""
    states = CHANNELS + ['Start', 'Convert', 'Null']
    trans = defaultdict(lambda: defaultdict(int))

    for _, row in df.iterrows():
        journey = row['journey']
        converted = row['converted']

        trans['Start'][journey[0]] += 1
        for i in range(len(journey) - 1):
            trans[journey[i]][journey[i+1]] += 1
        last = journey[-1]
        if converted:
            trans[last]['Convert'] += 1
        else:
            trans[last]['Null'] += 1
    return trans

def transition_probs(trans):
    """Normalise transition counts to probabilities."""
    probs = {}
    for state, nexts in trans.items():
        total = sum(nexts.values())
        probs[state] = {k: v/total for k, v in nexts.items()}
    return probs

def conv_probability(probs, removed_channel=None, n_steps=20):
    """Simulate conversion probability via random walks."""
    np.random.seed(42)
    n_sims = 5000
    conversions = 0

    for _ in range(n_sims):
        state = 'Start'
        for _ in range(n_steps):
            if state not in probs:
                break
            next_states = probs[state].copy()
            # Remove channel: redirect its traffic proportionally
            if removed_channel and removed_channel in next_states:
                removed_prob = next_states.pop(removed_channel)
                total_remaining = sum(next_states.values())
                if total_remaining > 0:
                    next_states = {k: v/total_remaining for k, v in next_states.items()}
                else:
                    next_states['Null'] = 1.0

            states_list = list(next_states.keys())
            prob_list   = list(next_states.values())
            if not states_list:
                break
            state = np.random.choice(states_list, p=prob_list)
            if state == 'Convert':
                conversions += 1
                break
            elif state == 'Null':
                break
    return conversions / n_sims

print('Building transition matrix...')
trans = build_transition_matrix(df)
probs = transition_probs(trans)

print('Computing baseline conversion probability...')
baseline_conv = conv_probability(probs)
print(f'Baseline conversion probability: {baseline_conv:.4f}')

print('Computing removal effects for each channel...')
removal_effects = {}
for ch in CHANNELS:
    conv_without = conv_probability(probs, removed_channel=ch)
    removal_effects[ch] = max(0, baseline_conv - conv_without)
    print(f'  {ch:<22}: removal effect = {removal_effects[ch]:.4f}')

# Normalise to revenue
total_removal = sum(removal_effects.values())
markov_rev = {ch: (v/total_removal) * total_converted_rev
              for ch, v in removal_effects.items()}

markov_df = pd.DataFrame({
    'Channel': [c.replace('_',' ') for c in CHANNELS],
    'Removal Effect': [removal_effects[c] for c in CHANNELS],
    'Revenue Attribution (£)': [markov_rev[c] for c in CHANNELS],
    '% Share': [markov_rev[c]/total_converted_rev*100 for c in CHANNELS]
}).sort_values('% Share', ascending=False)

print('\nMarkov Chain Attribution:')
print(markov_df.round(2).to_string(index=False))

---
## 6. Model Comparison

How different are the attribution models? The gap between Last Touch and Shapley/Markov tells you how much budget decisions are being distorted by naive attribution.

In [ ]:
# Compile all models
all_models = pd.DataFrame({
    'Channel'    : [c.replace('_',' ') for c in CHANNELS],
    'Last Touch' : [last_touch[c]/sum(last_touch.values())*100 for c in CHANNELS],
    'First Touch': [first_touch[c]/sum(first_touch.values())*100 for c in CHANNELS],
    'Linear'     : [linear[c]/sum(linear.values())*100 for c in CHANNELS],
    'Time Decay' : [time_decay[c]/sum(time_decay.values())*100 for c in CHANNELS],
    'Shapley'    : [shapley_rev[c]/total_converted_rev*100 for c in CHANNELS],
    'Markov'     : [markov_rev[c]/total_converted_rev*100 for c in CHANNELS],
})
all_models = all_models.sort_values('Shapley', ascending=False)

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

heat_data = all_models.set_index('Channel')[['Last Touch','First Touch','Linear','Time Decay','Shapley','Markov']]
sns.heatmap(heat_data, ax=axes[0], annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': '% Revenue Share'})
axes[0].set_title('Attribution % Share by Model — All Channels', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')

# Grouped bar — Last Touch vs Shapley vs Markov
x = np.arange(len(all_models))
w = 0.25
axes[1].bar(x - w, all_models['Last Touch'],  w, label='Last Touch',  color=PALETTE[6], alpha=0.9)
axes[1].bar(x,     all_models['Shapley'],     w, label='Shapley',     color=PALETTE[0], alpha=0.9)
axes[1].bar(x + w, all_models['Markov'],      w, label='Markov',      color=PALETTE[2], alpha=0.9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_models['Channel'], rotation=35, ha='right', fontsize=9)
axes[1].set_title('Last Touch vs Shapley vs Markov Attribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('% Revenue Share')
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/claude/mta_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(all_models.round(1).to_string(index=False))

---
## 7. Budget Implications

Different attribution models lead to **very different budget decisions**. Here we quantify how much budget would shift if a team moved from Last Touch to Shapley attribution.

In [ ]:
# Simulate budget based on attribution share
total_budget = sum(channel_props[c]['avg_spend'] for c in CHANNELS)

budget_df = all_models[['Channel','Last Touch','Shapley']].copy()
budget_df['Budget Last Touch (£K)'] = (budget_df['Last Touch'] / 100 * total_budget).round(1)
budget_df['Budget Shapley (£K)']    = (budget_df['Shapley']    / 100 * total_budget).round(1)
budget_df['Shift (£K)'] = (budget_df['Budget Shapley (£K)'] - budget_df['Budget Last Touch (£K)']).round(1)
budget_df['Direction']  = budget_df['Shift (£K)'].apply(lambda x: 'Increase' if x > 0 else 'Decrease')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Budget shift waterfall
colors_shift = [PALETTE[2] if v > 0 else PALETTE[6] for v in budget_df['Shift (£K)']]
axes[0].barh(budget_df['Channel'], budget_df['Shift (£K)'],
             color=colors_shift, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Budget Shift: Last Touch → Shapley Attribution\n(+ = increase, - = decrease)',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Budget Change (£K)')
axes[0].invert_yaxis()

# Side by side budget
x = np.arange(len(budget_df))
axes[1].bar(x - 0.2, budget_df['Budget Last Touch (£K)'], 0.4,
            label='Last Touch', color=PALETTE[6], alpha=0.9)
axes[1].bar(x + 0.2, budget_df['Budget Shapley (£K)'], 0.4,
            label='Shapley', color=PALETTE[0], alpha=0.9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(budget_df['Channel'], rotation=35, ha='right', fontsize=9)
axes[1].set_title('Budget Allocation: Last Touch vs Shapley', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Budget (£K)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/claude/mta_budget.png', dpi=150, bbox_inches='tight')
plt.show()

print('Budget reallocation summary:')
print(budget_df[['Channel','Budget Last Touch (£K)','Budget Shapley (£K)','Shift (£K)','Direction']].to_string(index=False))

---
## 8. Key Takeaways

| Model | Pros | Cons | When to Use |
|-------|------|------|-------------|
| **Last Touch** | Simple, fast | Ignores awareness channels | Never for strategy |
| **First Touch** | Simple, fast | Ignores conversion channels | Brand measurement only |
| **Linear** | Fair, transparent | Ignores channel effectiveness | Good baseline |
| **Time Decay** | Recency-weighted | Arbitrary half-life | Short purchase cycles |
| **Shapley** | Theoretically fair, captures interactions | Computationally expensive | Gold standard |
| **Markov** | Journey-level, removal effect | Simulation-based variance | Complements Shapley |

### Real-world lessons

**1. Last touch systematically overstates onsite channels.**  
Loyalty coupons and in-store displays appear late in the journey — so Last Touch gives them all the credit. But without the TV and digital ads that drove awareness, those customers wouldn't have been in the store at all.

**2. Shapley + Markov should agree directionally.**  
If TV is undervalued by Last Touch, both Shapley and Markov will show it — they use different mathematics but both capture marginal contribution. When they disagree strongly, investigate your data pipeline.

**3. MTA and MMM answer different questions.**  
MTA works at the individual journey level — great for digital channels where you can track clicks and cookies. MMM works at the aggregate level — necessary for TV, OOH, and Radio where individual tracking isn't possible. The two models are **complementary**, not competing. In production, run both and triangulate.

**4. The budget shift is the deliverable.**  
The chart that matters most is the budget reallocation. If Last Touch says spend 40% on Loyalty Coupons but Shapley says 18%, that's a £Xm decision. MTA's value is in making that conversation data-driven.

---

### Next in this series
- 📈 [Market Mix Modeling](../01_market_mix_modeling/) — aggregate-level attribution with external variables
- 💰 Price Elasticity Modelling — coming soon
- 👥 Customer Segmentation — coming soon

---
*Built by Shashank Paliwal — [LinkedIn](https://linkedin.com/in/shashank-paliwal-ba1ba171) | [Medium](#)*  
*All data in this notebook is synthetic and generated for demonstration purposes.*